# Smart Motrex Yards — Notebook First

**Phase 1:** Template 55 on full group + polygon match → vehicles inside yard geofences NOW (seconds).

**Phase 2:** Template 58 direct exec (**no remoteExec**) only for inside vehicles → Time In → live duration.

**Neon:** `MotrexTransferDB` → `motrex_yards`

Quick test: **Motrex - KBK 012E** (cell 4). Full flow: cell 5 (inside-now) → cell 6 (template 58 each) → cell 7 (Neon).

CLI equivalent: `npm run yards:smart-kbk` or `npm run yards:smart`

In [ ]:
import json
import os
import re
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import requests

# Load .env from project root
for env_name in (".env.local", ".env"):
    env_path = Path.cwd() / env_name
    if env_path.exists():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, val = line.partition("=")
            key = key.strip()
            val = val.strip().strip('"').strip("'")
            os.environ.setdefault(key, val)

WIALON_TOKEN = os.environ.get("WIALON_TOKEN", "").strip()
DATABASE_URL = os.environ.get("MotrexTransferDB", "").strip().strip('"')
assert WIALON_TOKEN, "WIALON_TOKEN missing in .env"
assert DATABASE_URL, "MotrexTransferDB missing in .env"

WIALON_URL = "https://hst-api.wialon.com/wialon/ajax.html"
REPORT_RESOURCE_ID = 26054231
TEMPLATE_ID = 58  # Motrex Geofence Visits
GROUP_ID = 26659458
TARGET_REGISTRATION = "KBK 012E"

YARDS_GEOFENCES = {
    "Motrex Mikindani Yard",
    "Vipingo Main yard",
    "MCL PARKING VIPINGO",
    "Vipingo_Loading_Zone",
    "MOTREX NO GO ZONE_RED ZONE",
    "Tororo Cement(Parking)",
    "Tororo Cement(Uganda)",
    "Mombasa Cement Athi River",
}

EAT = timezone(timedelta(hours=3))
now_eat = datetime.now(EAT)
from_dt = now_eat - timedelta(days=30)
from_ts = int(from_dt.timestamp())
to_ts = int(now_eat.timestamp())
report_date = now_eat.strftime("%Y-%m-%d")
print(f"Range: {from_dt:%Y-%m-%d %H:%M} EAT -> {now_eat:%Y-%m-%d %H:%M} EAT")
print(f"Report date: {report_date}")

In [ ]:
session = requests.Session()


def wialon_call(svc: str, params: dict, sid: str, retries: int = 5):
    last_err = None
    for attempt in range(retries):
        resp = session.post(
            WIALON_URL,
            data={"svc": svc, "params": json.dumps(params), "sid": sid},
            timeout=300,
        )
        resp.raise_for_status()
        data = resp.json()
        if isinstance(data, dict) and data.get("error"):
            last_err = RuntimeError(f"Wialon error {data['error']} for {svc}: {data}")
            time.sleep(2 ** attempt)
            continue
        return data
    raise last_err


def wialon_login(token: str) -> str:
    data = wialon_call("token/login", {"token": token}, "")
    sid = data.get("eid")
    if not sid:
        raise RuntimeError(f"Login failed: {data}")
    return sid


def cell_text(cell) -> str:
    if isinstance(cell, dict):
        return str(cell.get("t") or "").strip()
    return str(cell or "").strip()


def exec_report_direct(resource_id: int, template_id: int, object_id: int, sid: str) -> dict:
    wialon_call("report/cleanup_result", {}, sid)
    return wialon_call(
        "report/exec_report",
        {
            "reportResourceId": resource_id,
            "reportTemplateId": template_id,
            "reportObjectId": object_id,
            "reportObjectSecId": 0,
            "interval": {"flags": 0, "from": from_ts, "to": to_ts},
        },
        sid,
    )


def get_table_subrows(table_index: int, row_index: int, row_count: int, sid: str) -> list:
    if row_count <= 0:
        return []
    return wialon_call(
        "report/get_result_subrows",
        {"tableIndex": table_index, "rowIndex": row_index, "indexFrom": 0, "indexTo": row_count - 1},
        sid,
    )


def pick_zones_visit_table(tables: list) -> int | None:
    for i, t in enumerate(tables):
        headers = [str(h).lower() for h in (t.get("header") or [])]
        joined = " ".join(headers)
        if "geofence" in joined and "time in" in joined:
            return i
    for i, t in enumerate(tables):
        if int(t.get("rows", 0) or 0) > 0:
            return i
    return None


def normalize_geofence(name: str) -> str | None:
    raw = re.sub(r"\s+", " ", str(name or "").strip())
    if not raw:
        return None
    low = raw.lower()
    for g in YARDS_GEOFENCES:
        if g.lower() == low:
            return g
    aliases = [
        (r"mcl\s*parking\s*vipingo", "MCL PARKING VIPINGO"),
        (r"vipingo\s*main\s*yard", "Vipingo Main yard"),
        (r"vipingo[\s_]*loading", "Vipingo_Loading_Zone"),
    ]
    for pattern, canonical in aliases:
        if re.search(pattern, raw, re.I):
            return canonical
    return None


def registration_label(value: str) -> str:
    return re.sub(r"^Motrex\s*-\s*", "", str(value or "").strip(), flags=re.I).strip()


sid = wialon_login(WIALON_TOKEN)
print("Logged in, sid:", sid[:12], "...")

In [ ]:
def find_unit_id_by_registration(sid: str, registration: str) -> tuple[int, str]:
    group = wialon_call("core/search_item", {"id": GROUP_ID, "flags": 1}, sid)
    unit_ids = []
    for key in ("u", "units"):
        candidate = group.get(key) or (group.get("item") or {}).get(key)
        if isinstance(candidate, list):
            unit_ids = [int(x) for x in candidate if str(x).isdigit()]
            break
        if isinstance(candidate, dict):
            unit_ids = [int(k) for k in candidate.keys() if str(k).isdigit()]
            break
    print(f"Group has {len(unit_ids)} units")
    reg_upper = registration.upper()
    for uid in unit_ids:
        item = wialon_call("core/search_item", {"id": uid, "flags": 1}, sid)
        name = str(item.get("nm") or item.get("name") or "")
        if reg_upper in name.upper():
            return uid, name
    raise RuntimeError(f"Unit not found for registration {registration}")


unit_id, unit_name = find_unit_id_by_registration(sid, TARGET_REGISTRATION)
print(f"Found unit_id={unit_id} name={unit_name}")

In [ ]:
def fetch_visits_for_unit(sid: str, unit_id: int) -> pd.DataFrame:
    t0 = time.time()
    print(f"Executing template {TEMPLATE_ID} direct exec for unit {unit_id}...")
    exec_result = exec_report_direct(REPORT_RESOURCE_ID, TEMPLATE_ID, unit_id, sid)
    tables = exec_result.get("reportResult", {}).get("tables", []) or []
    print("Tables:", [{"i": i, "rows": t.get("rows"), "header": t.get("header")} for i, t in enumerate(tables)])
    table_idx = pick_zones_visit_table(tables)
    if table_idx is None:
        print(f"No visit table in {time.time() - t0:.1f}s")
        return pd.DataFrame()
    table = tables[table_idx]
    headers = table.get("header") or []
    parent_rows = int(table.get("rows", 0) or 0)
    visits = []
    for parent_idx in range(parent_rows):
        subrows = get_table_subrows(table_idx, parent_idx, 5000, sid)
        if not isinstance(subrows, list):
            continue
        vehicle = unit_name
        for sub in subrows:
            cells = sub.get("c", []) if isinstance(sub, dict) else []
            # Subrows include Grouping as first column (5 cells total).
            off = 1 if len(cells) >= 5 else 0
            geofence = normalize_geofence(cell_text(cells[off] if len(cells) > off else ""))
            if not geofence:
                continue
            visits.append({
                "Grouping": vehicle,
                "Vehicle": vehicle,
                "Geofence": geofence,
                "Time in": cell_text(cells[off + 1] if len(cells) > off + 1 else ""),
                "Time out": cell_text(cells[off + 2] if len(cells) > off + 2 else ""),
                "Duration in": cell_text(cells[off + 3] if len(cells) > off + 3 else ""),
            })
    elapsed = time.time() - t0
    print(f"Fetched {len(visits)} visits in {elapsed:.1f}s")
    return pd.DataFrame(visits)


visits_df = fetch_visits_for_unit(sid, unit_id)
visits_df.head(10)

In [ ]:
def save_to_neon(df: pd.DataFrame) -> int:
    try:
        import psycopg2
    except ImportError:
        import subprocess
        subprocess.check_call(["pip", "install", "psycopg2-binary", "-q"])
        import psycopg2

    reg = registration_label(unit_name)
    now_label = now_eat.strftime("%Y-%m-%d %H:%M:%S") + " EAT"
    conn = psycopg2.connect(DATABASE_URL)
    conn.autocommit = False
    try:
        with conn.cursor() as cur:
            cur.execute("DELETE FROM motrex_yards WHERE report_date = %s", (report_date,))
            inserted = 0
            for _, row in df.iterrows():
                time_out = str(row.get("Time out") or "").strip()
                status = "Inside" if not time_out or time_out in ("—", "-", "0") else "Out"
                cur.execute(
                    """
                    INSERT INTO motrex_yards
                      (report_date, registration_number, vehicle, geofence, time_in, time_out,
                       duration_seconds, status, last_execution_time, raw_row, updated_at)
                    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,NOW())
                    """,
                    (
                        report_date,
                        reg,
                        reg,
                        row["Geofence"],
                        row.get("Time in"),
                        row.get("Time out") or None,
                        None,
                        status,
                        now_label,
                        json.dumps(row.to_dict()),
                    ),
                )
                inserted += 1
        conn.commit()
        return inserted
    finally:
        conn.close()


if visits_df.empty:
    print("No visits to save.")
else:
    n = save_to_neon(visits_df)
    print(f"Saved {n} rows to motrex_yards for {report_date}")
    print(f"Registration: {registration_label(unit_name)}")
    print(visits_df[["Geofence", "Time in", "Time out"]].head())

In [ ]:
# Verify Neon read-back
try:
    import psycopg2
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "psycopg2-binary", "-q"])
    import psycopg2

conn = psycopg2.connect(DATABASE_URL)
with conn.cursor() as cur:
    cur.execute(
        "SELECT registration_number, geofence, time_in, time_out FROM motrex_yards WHERE report_date = %s LIMIT 5",
        (report_date,),
    )
    rows = cur.fetchall()
conn.close()
print("Neon sample rows:")
for r in rows:
    print(r)